In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
import scanpy as sc
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from scipy.sparse import issparse
import SpaDiff as sd

In [ ]:
parser = sd.get_args() 
args, unknown = parser.parse_known_args() 
args.n_clusters = 7
args.domains = 7
print(args.random_seed)
torch.manual_seed(args.random_seed)
torch.cuda.manual_seed_all(args.random_seed)
np.random.seed(args.random_seed)
random.seed(args.random_seed)
torch.backends.cudnn.deterministic = True
if args.cuda >= 0 and torch.cuda.is_available():
    device = torch.device(f"cuda:{args.cuda}")
    torch.cuda.manual_seed_all(args.random_seed)
    print(f"Using GPU: {torch.cuda.get_device_name(args.cuda)}")
else:
    device = torch.device("cpu")
    print("Using CPU.")

In [ ]:
st_samples = ["H_1","H_2","H_3"]  
batch_list = []
for i,sample in enumerate(st_samples):  
    st_adata = sc.read_h5ad("../0_data/case3/"+sample+".h5ad") 
    st_adata.var_names_make_unique()
    sc.pp.normalize_total(st_adata)
    sc.pp.log1p(st_adata)
    st_adata.obs["batch_name"] = sample
    st_adata.obs_names = [x+'-'+str(i) for x in st_adata.obs_names]
    
    batch_list.append(st_adata)

adata = sc.concat(batch_list)

In [ ]:
adata

In [ ]:
sc.pp.highly_variable_genes(adata,subset=True, flavor="seurat_v3", n_top_genes=3000,batch_key="batch_name")
adata, HL = sd.spatial_rec_multi(adata, normalize_total=True, alpha=args.rec_alpha)

In [ ]:
sc.pp.pca(adata, n_comps=50)
features = torch.tensor(adata.obsm['X_pca'].toarray() if issparse(adata.obsm['X_pca']) else np.array(adata.obsm['X_pca'])).float()

In [ ]:
clf=sd.DEC( features, HL, 
              node_width=features.shape[1],
              device = device,
              opt="adam",
              args=args)
a = clf.fit(features, HL)
y_pred, prob, z=clf.predict()
adata.obsm['z'] = z

In [ ]:
adata = sd.adjust_louvain_resolution(adata, args.domains, use_rep='z')

In [ ]:
Batch_list = []
for sample in st_samples:
    Batch_list.append(adata[adata.obs['batch_name'] == sample])

In [ ]:
puriety_list = []
for i in range(len(st_samples)):
    p = sd.cal_purity(Batch_list[i].obs['pathologist'], Batch_list[i].obs['louvain'])
    p = round(p, 3)
    puriety_list.append(p)
print(puriety_list)

purity = sd.cal_purity(adata.obs['pathologist'], adata.obs['louvain'])
purity = round(purity, 3)
print(purity)

In [ ]:
plot_color =["#7495D3","#59BE86","#FEB915","#C798EE","#6D1A9C","#F56867","#D1D1D1"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, ad in enumerate(Batch_list):
    sc.pl.spatial(
        ad,
        color="louvain",
        ax=axes[i],
        show=False,
        spot_size=280,
        palette=plot_color,
        # legend_loc=None, 
        # frameon=False,
        title=""
    )
plt.tight_layout()
plt.show()

In [ ]:
sc.pp.neighbors(adata,use_rep='z')
sc.tl.umap(adata)
sc.pl.umap(adata, 
           color="batch_name", 
           size = 20,
           legend_fontsize=14,
           legend_fontoutline=4,
           )
sc.pl.umap(adata, 
           color="louvain", 
           size = 20,
           legend_fontsize=14,
           legend_fontoutline=4,
           )